# Topic Modeling pe Wikipedia: Persoana A (date + LSA)

Acest notebook acoperă **doar partea persoanei A**:

1. Alegerea a 3 domenii și descărcarea a câte 10 articole Wikipedia pentru fiecare
2. Preprocesare (lowercase, tokenizare, lemmatizare, stop-words) și matricile **BoW** și **TF-IDF**
3. **Exportul** datelor într-un format fix, pentru persoana B (folderul `shared/`)
4. **LSA** (`TruncatedSVD`) pe BoW și pe TF-IDF, pentru **k = 3 și k = 5**:
   * top 10 cuvinte per topic
   * atribuirea documentelor la topicul dominant (scor absolut maxim)
   * scatter plot-uri (cuvinte + documente)
   * scoruri de coerență Gensim
5. Șablon pentru Partea 4 (răspunsurile la cele 5 întrebări, doar pentru LSA)

**Ordinea de lucru:** rulați secțiunile 0 → 3 prima dată și trimiteți folderul `shared/` persoanei B. Restul (LSA) se poate face după.

In [1]:
%pip install -q scikit-learn gensim nltk requests numpy pandas scipy plotly

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import re
import time
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from scipy import sparse
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import adjusted_rand_score

# ---- parametri comuni cu persoana B (nu le schimbați fără să vorbiți) ----
RANDOM_STATE = 42
K_VALUES = (3, 5)
N_TOP_WORDS = 10
CHUNK_SIZE = 100          # tokeni per pseudo-paragraf (corpus de referință pentru coerență)
MIN_DF, MAX_DF = 2, 0.7   # filtre de vocabular

CACHE_FILE = Path("wiki_corpus_30.json")
OUT_DIR = Path("shared")            # aici se exportă tot ce primește persoana B
(OUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
SHOW_PLOTS = True                   # False = doar salvează graficele ca HTML, fără să le afișeze

## 0. Date: 10 articole Wikipedia per domeniu

In [3]:
CORPUS_SPEC = {
    "green_energy": [
        "Solar power", "Wind power", "Hydroelectricity", "Geothermal energy",
        "Biofuel", "Tidal power", "Renewable energy", "Wave power",
        "Green hydrogen", "Photovoltaics",
    ],
    "animals": [
        "Lion", "Elephant", "Blue whale", "Gray wolf", "Bald eagle",
        "Giant panda", "Tiger", "Polar bear", "Bottlenose dolphin", "Gorilla",
    ],
    "roman_empire": [
        "Roman Empire", "Augustus", "Nero", "Constantine the Great", "Trajan",
        "Hadrian", "Diocletian", "Marcus Aurelius",
        "Fall of the Western Roman Empire", "Roman army",
    ],
}

WIKI_API = "https://en.wikipedia.org/w/api.php"
# Wikipedia cere o adresă de contact în User-Agent: puneți emailul vostru
HEADERS = {"User-Agent": "TopicModelingAssignment/1.0 (student project; your_email@example.com)"}
TAIL_SECTIONS = (
    "See also|References|Notes|Footnotes|External links|Further reading|"
    "Bibliography|Sources|Citations|Works cited"
)


def fetch_article(title, max_retries=6):
    params = {"action": "query", "prop": "extracts", "explaintext": 1,
              "redirects": 1, "titles": title, "format": "json"}
    for attempt in range(max_retries):
        r = requests.get(WIKI_API, params=params, headers=HEADERS, timeout=30)
        if r.status_code == 429 or r.status_code >= 500:      # rate limit / server ocupat
            try:
                wait = int(r.headers.get("Retry-After", 0))
            except ValueError:
                wait = 0
            wait = wait or 10 * 2 ** attempt
            print(f"  HTTP {r.status_code} pentru '{title}': aștept {wait}s "
                  f"(încercarea {attempt + 1}/{max_retries})")
            time.sleep(wait)
            continue
        r.raise_for_status()
        text = next(iter(r.json()["query"]["pages"].values())).get("extract", "")
        if not text:
            raise ValueError(f"Nu s-a returnat text pentru '{title}'")
        text = re.split(rf"\n==+\s*(?:{TAIL_SECTIONS})\s*==+", text, maxsplit=1)[0]
        text = re.sub(r"^=+\s*.*?\s*=+\s*$", " ", text, flags=re.MULTILINE)
        return text
    raise RuntimeError(f"Renunț la '{title}' după {max_retries} încercări; "
                       "așteptați câteva minute și rulați din nou celula.")


def load_corpus(pause=3):
    # reluabil: articolele deja descărcate sunt păstrate și sărite
    data = json.loads(CACHE_FILE.read_text(encoding="utf-8")) if CACHE_FILE.exists() else []
    done = {d["title"] for d in data}
    for subject, subject_titles in CORPUS_SPEC.items():
        for t in subject_titles:
            if t in done:
                continue
            print("Descarc:", t)
            data.append({"title": t, "topic": subject, "text": fetch_article(t)})
            CACHE_FILE.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
            time.sleep(pause)
    order = {t: i for i, t in enumerate(x for ts in CORPUS_SPEC.values() for x in ts)}
    data = sorted((d for d in data if d["title"] in order), key=lambda d: order[d["title"]])
    return ([d["title"] for d in data], [d["text"] for d in data],
            [d["topic"] for d in data])


titles, raw_texts, subjects = load_corpus()
classes = sorted(set(subjects))
true_idx = np.array([classes.index(s) for s in subjects])
print(f"{len(raw_texts)} documente | domenii: {classes}")
print("Caractere per document:", [len(t) for t in raw_texts])

30 documente | domenii: ['animals', 'green_energy', 'roman_empire']
Caractere per document: [28138, 34633, 28613, 22342, 25084, 22439, 57465, 37829, 35094, 52428, 51176, 54578, 34528, 54901, 58062, 37800, 49293, 47252, 38963, 35678, 97821, 81664, 44749, 4641, 2067, 65603, 70664, 2741, 91983, 28020]


## 1. Preprocesare, Bag-of-Words și TF-IDF

In [4]:
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

EXTRA_STOP = {"also", "may", "one", "two", "first", "new", "use", "used", "many",
              "however", "include", "including", "well", "often", "make", "see",
              "later", "several", "since", "like", "become", "within", "year"}
STOP = set(stopwords.words("english")) | EXTRA_STOP

#lematizarea aduce cuvintele la forma de baza/infinitiv
lemmatizer = WordNetLemmatizer()


def preprocess(text):
    tokens = re.findall(r"[a-z]+", text.lower())      # elimină cifre și punctuație
    out = []
    for tok in tokens:
        if len(tok) < 3 or tok in STOP:
            continue
        lemma = lemmatizer.lemmatize(lemmatizer.lemmatize(tok, pos="v"), pos="n")
        if len(lemma) >= 3 and lemma not in STOP:
            out.append(lemma)
    return out


docs_tokens = [preprocess(t) for t in raw_texts]
docs_str = [" ".join(t) for t in docs_tokens]
print("Tokeni per document:", [len(t) for t in docs_tokens])

# Aceiași tokeni și aceleași filtre de vocabular pentru ambele reprezentări
common = dict(tokenizer=str.split, preprocessor=None, lowercase=False,
              token_pattern=None, min_df=MIN_DF, max_df=MAX_DF)
bow_vec, tfidf_vec = CountVectorizer(**common), TfidfVectorizer(**common)
X_bow = bow_vec.fit_transform(docs_str)
X_tfidf = tfidf_vec.fit_transform(docs_str)

feats = list(bow_vec.get_feature_names_out())
assert feats == list(tfidf_vec.get_feature_names_out()), "vocabularele diferă"
MATRICES = {"BoW": X_bow, "TF-IDF": X_tfidf}
print(f"BoW: {X_bow.shape} | TF-IDF: {X_tfidf.shape} | vocabular: {len(feats)} cuvinte")

Tokeni per document: [2518, 3064, 2406, 1918, 2138, 2008, 5076, 2154, 3050, 4449, 4554, 4632, 3022, 4957, 5205, 3303, 4292, 4359, 3455, 3009, 8666, 7095, 3831, 367, 180, 5721, 5855, 236, 7767, 2377]
BoW: (30, 5854) | TF-IDF: (30, 5854) | vocabular: 5854 cuvinte


## 2. Export pentru persoana B (interfața comună)

Se salvează în `shared/`:

| Fișier | Conținut |
|---|---|
| `corpus.csv` | coloanele `title, text, subject, tokens` (`tokens` = tokenii preprocesați, separați prin spațiu) |
| `bow.npz`, `tfidf.npz` | matricile documente × vocabular (`scipy.sparse`) |
| `vocab.json` | vocabularul, în ordinea coloanelor matricilor |
| `chunks.json`, `gensim_dictionary.dict` | corpusul de referință pentru coerența Gensim (pseudo-paragrafe) și dicționarul lui |
| `config.json` | `random_state`, valorile lui k, număr de cuvinte, filtre |

Persoana B încarcă totul așa:

```python
import json, pandas as pd
from scipy import sparse
from gensim.corpora import Dictionary

df = pd.read_csv("shared/corpus.csv")
X_bow = sparse.load_npz("shared/bow.npz")
X_tfidf = sparse.load_npz("shared/tfidf.npz")
vocab = json.load(open("shared/vocab.json"))
chunks = json.load(open("shared/chunks.json"))
dictionary = Dictionary.load("shared/gensim_dictionary.dict")
cfg = json.load(open("shared/config.json"))
```

In [5]:
def chunk_tokens(docs, size=CHUNK_SIZE):
    """Împarte fiecare document în bucăți de `size` tokeni (coada <size/2 se ignoră)."""
    chunks = []
    for toks in docs:
        for i in range(0, len(toks), size):
            piece = toks[i:i + size]
            if len(piece) >= size // 2:
                chunks.append(piece)
    return chunks


chunk_texts = chunk_tokens(docs_tokens)
dictionary = Dictionary(chunk_texts)
print(f"Corpus de referință pentru coerență: {len(chunk_texts)} pseudo-paragrafe, "
      f"{len(dictionary)} cuvinte în dicționar")

# ---- scriere fișiere ----
pd.DataFrame({"title": titles, "text": raw_texts, "subject": subjects,
              "tokens": docs_str}).to_csv(OUT_DIR / "corpus.csv", index=False)
sparse.save_npz(OUT_DIR / "bow.npz", X_bow.tocsr())
sparse.save_npz(OUT_DIR / "tfidf.npz", X_tfidf.tocsr())
(OUT_DIR / "vocab.json").write_text(json.dumps(feats, ensure_ascii=False), encoding="utf-8")
(OUT_DIR / "chunks.json").write_text(json.dumps(chunk_texts), encoding="utf-8")
dictionary.save(str(OUT_DIR / "gensim_dictionary.dict"))
(OUT_DIR / "config.json").write_text(json.dumps({
    "random_state": RANDOM_STATE, "k_values": list(K_VALUES), "n_top_words": N_TOP_WORDS,
    "chunk_size": CHUNK_SIZE, "min_df": MIN_DF, "max_df": MAX_DF,
}, indent=2), encoding="utf-8")

# ---- verificare: se pot reciti fișierele? ----
chk = pd.read_csv(OUT_DIR / "corpus.csv")
assert list(chk.columns) == ["title", "text", "subject", "tokens"] and len(chk) == len(titles)
assert sparse.load_npz(OUT_DIR / "bow.npz").shape == X_bow.shape
assert sparse.load_npz(OUT_DIR / "tfidf.npz").shape == X_tfidf.shape
print("Export OK ->", sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))

Corpus de referință pentru coerență: 1119 pseudo-paragrafe, 13060 cuvinte în dicționar
Export OK -> ['bow.npz', 'chunks.json', 'config.json', 'corpus.csv', 'gensim_dictionary.dict', 'tfidf.npz', 'vocab.json']


## 3. LSA (`TruncatedSVD`) pe BoW și TF-IDF, k = 3 și k = 5

Pentru fiecare combinație (matrice, k) se păstrează:
* `doc_topics` = U·Σ (coordonatele documentelor în spațiul topicurilor)
* `components` = Vᵀ (încărcarea fiecărui cuvânt pe fiecare topic)
* `top_idx` = indicii celor 10 cuvinte cu cea mai mare încărcare pozitivă pe fiecare topic

Semnul SVD este arbitrar, așa că fiecare componentă este orientată astfel încât încărcarea ei cea mai mare în valoare absolută să fie pozitivă.

In [6]:
def top_idx(components, n=N_TOP_WORDS):
    return [np.argsort(row)[::-1][:n] for row in components]


def run_lsa(X, name, k):
    svd = TruncatedSVD(n_components=k, random_state=RANDOM_STATE)
    D = svd.fit_transform(X)              # U * Sigma
    C = svd.components_.copy()            # V^T
    for j in range(k):                    # semn arbitrar: încărcarea maximă devine pozitivă
        if C[j, np.argmax(np.abs(C[j]))] < 0:
            C[j] *= -1
            D[:, j] *= -1
    return dict(name=name, k=k, doc_topics=D, components=C, top_idx=top_idx(C),
                sv=svd.singular_values_, evr=svd.explained_variance_ratio_)


def topic_words(res):
    return [[feats[i] for i in idx] for idx in res["top_idx"]]


results = {}
for k in K_VALUES:
    for mname, X in MATRICES.items():
        name = f"LSA ({mname}, k={k})"
        results[name] = run_lsa(X, name, k)

for name, res in results.items():
    print(f"\n=== {name} | varianță explicată: {np.round(res['evr'], 3)} ===")
    for j, words in enumerate(topic_words(res)):
        print(f"  Topic {j}: {', '.join(words)}")


=== LSA (BoW, k=3) | varianță explicată: [0.07  0.105 0.089] ===
  Topic 0: roman, empire, augustus, rome, emperor, wolf, hadrian, bear, diocletian, octavian
  Topic 1: wolf, bear, lion, polar, prey, energy, specie, eagle, tiger, male
  Topic 2: energy, solar, wind, renewable, electricity, hydrogen, cost, fuel, capacity, technology

=== LSA (TF-IDF, k=3) | varianță explicată: [0.039 0.104 0.041] ===
  Topic 0: roman, empire, energy, emperor, solar, army, hadrian, rome, diocletian, wind
  Topic 1: energy, solar, wind, electricity, renewable, geothermal, tidal, hydrogen, cost, fuel
  Topic 2: lion, tiger, elephant, gorilla, dolphin, bear, polar, eagle, panda, male

=== LSA (BoW, k=5) | varianță explicată: [0.07  0.105 0.089 0.086 0.07 ] ===
  Topic 0: roman, empire, augustus, rome, emperor, wolf, hadrian, bear, diocletian, octavian
  Topic 1: wolf, bear, lion, polar, prey, energy, specie, eagle, tiger, male
  Topic 2: energy, solar, wind, renewable, electricity, hydrogen, cost, fuel, ca

## 4. Atribuirea documentelor la topicuri

Metoda cerută: fiecare document merge la topicul cu **cel mai mare scor absolut** (`argmax |U·Σ|`).

Atenție: la LSA, prima componentă captează în mare parte "cât de mult text" are documentul și este pozitivă pentru toate documentele, deci `argmax` absolut poate pune aproape tot corpusul pe topicul 0. Ca să se vadă dacă asta se întâmplă, tabelul de mai jos arată și **mărimea celui mai mare grup**. Ca variantă de comparație se dă și KMeans pe vectorii documentelor normalizați (k clustere).

Pentru că avem 3 domenii, dar și k = 5 topicuri, măsura principală este **puritatea**: fracția documentelor care aparțin domeniului majoritar al topicului în care au fost puse (1.0 = fiecare topic conține un singur domeniu).

In [7]:
def dominant_abs(res):
    return np.abs(res["doc_topics"]).argmax(axis=1)


def dominant_kmeans(res):
    D = res["doc_topics"]
    Dn = D / np.clip(np.linalg.norm(D, axis=1, keepdims=True), 1e-12, None)
    return KMeans(n_clusters=res["k"], n_init=10, random_state=RANDOM_STATE).fit_predict(Dn)


def purity(pred):
    total = 0
    for c in np.unique(pred):
        total += np.bincount(true_idx[pred == c], minlength=len(classes)).max()
    return total / len(pred)


assignments = {}
rows = []
for name, res in results.items():
    p_abs, p_km = dominant_abs(res), dominant_kmeans(res)
    assignments[name] = p_abs
    rows.append({"model": name,
                 "purity (abs)": purity(p_abs), "ARI (abs)": adjusted_rand_score(true_idx, p_abs),
                 "cel mai mare grup (abs)": int(np.bincount(p_abs).max()),
                 "purity (kmeans)": purity(p_km), "ARI (kmeans)": adjusted_rand_score(true_idx, p_km)})
assign_summary = pd.DataFrame(rows).set_index("model")
assign_summary.round(3)

,purity (abs),ARI (abs),cel mai mare grup (abs),purity (kmeans),ARI (kmeans)
model,,,,,
"LSA (BoW, k=3)",0.933,0.808,12,1.0,1.000
"LSA (TF-IDF, k=3)",1.000,1.000,10,1.0,1.000
"LSA (BoW, k=5)",0.967,0.745,11,1.0,0.870
"LSA (TF-IDF, k=5)",1.000,0.785,10,1.0,0.714


In [8]:
# Documentele din fiecare domeniu, pe topicurile în care au fost puse (metoda abs)
for name, pred in assignments.items():
    print(f"\n=== {name} ===")
    print(pd.crosstab(pd.Series(np.array(subjects), name="domeniu"),
                      pd.Series(pred, name="topic (abs)")))

# tabelul complet, document cu document
assign_df = pd.DataFrame({"title": titles, "subject": subjects,
                          **{name: pred for name, pred in assignments.items()}})
assign_df.to_csv(OUT_DIR / "lsa_assignments_A.csv", index=False)
assign_df


=== LSA (BoW, k=3) ===
topic (abs)    0  1   2
domeniu                
animals        2  8   0
green_energy   0  0  10
roman_empire  10  0   0

=== LSA (TF-IDF, k=3) ===
topic (abs)    0   1   2
domeniu                 
animals        0   0  10
green_energy   0  10   0
roman_empire  10   0   0

=== LSA (BoW, k=5) ===
topic (abs)    0  1   2  3  4
domeniu                      
animals        1  4   0  2  3
green_energy   0  0  10  0  0
roman_empire  10  0   0  0  0

=== LSA (TF-IDF, k=5) ===
topic (abs)   0  1   2  3  4
domeniu                     
animals       0  0  10  0  0
green_energy  0  8   0  0  2
roman_empire  7  0   0  3  0


,title,subject,"LSA (BoW, k=3)","LSA (TF-IDF, k=3)","LSA (BoW, k=5)","LSA (TF-IDF, k=5)"
0,Solar power,green_energy,2,1,2,1
1,Wind power,green_energy,2,1,2,1
2,Hydroelectricity,green_energy,2,1,2,1
3,Geothermal energy,green_energy,2,1,2,1
4,Biofuel,green_energy,2,1,2,1
5,Tidal power,green_energy,2,1,2,4
6,Renewable energy,green_energy,2,1,2,1
7,Wave power,green_energy,2,1,2,4
8,Green hydrogen,green_energy,2,1,2,1
9,Photovoltaics,green_energy,2,1,2,1


## 5. Scatter plot-uri: documente și top cuvinte în același spațiu

* **Documente** (cercuri), colorate după domeniul real Wikipedia, poziționate la (U·Σ) pe topicurile alese.
* **Top 10 cuvinte** ale fiecărui topic (romburi cu etichetă), poziționate la (V·Σ). Culoarea romburilor = topicul din care provin.
* Pentru fiecare model: un grafic 3D pe primele 3 topicuri și grafice 2D pe perechi de topicuri (k=5 are o pereche în plus, ca să apară toate cele 5 topicuri). Graficele sunt salvate și ca HTML în `shared/plots/`.

In [9]:
SUBJECT_COLORS = ["#1b9e77", "#d95f02", "#7570b3"]
TOPIC_COLORS = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00"]
DIM_SETS = {3: [(0, 1, 2), (1, 2)], 5: [(0, 1, 2), (1, 2), (3, 4)]}


def embed(res):
    idx = np.concatenate(res["top_idx"])
    word_topic = np.repeat(np.arange(res["k"]), N_TOP_WORDS)
    words = res["components"][:, idx].T * res["sv"]      # V * Sigma
    return res["doc_topics"], words, idx, word_topic


def plot_scatter(res, dims):
    docs, words, idx, word_topic = embed(res)
    tw = topic_words(res)
    three = len(dims) == 3
    Trace = go.Scatter3d if three else go.Scatter
    xyz = lambda M, mask: dict(zip("xyz", (M[mask, d] for d in dims)))
    fig = go.Figure()

    for c, cls in enumerate(classes):                    # documente: un trace per domeniu
        m = true_idx == c
        names = [titles[i] for i in np.where(m)[0]]
        fig.add_trace(Trace(**xyz(docs, m), mode="markers", text=names, hoverinfo="text",
                            name=f"documente: {cls}",
                            marker=dict(size=6 if three else 10, color=SUBJECT_COLORS[c],
                                        opacity=0.85, line=dict(width=1, color="black"))))

    for j in range(res["k"]):                            # cuvinte: un trace per topic
        m = word_topic == j
        fig.add_trace(Trace(**xyz(words, m), mode="markers+text",
                            text=[feats[i] for i in idx[m]], textposition="top center",
                            hoverinfo="text", name=f"top cuvinte: topic {j}",
                            marker=dict(size=5 if three else 9, symbol="diamond",
                                        color=TOPIC_COLORS[j % len(TOPIC_COLORS)],
                                        line=dict(width=1, color="black"))))

    ax = lambda d: dict(title=f"Topic {d}: {', '.join(tw[d][:3])}")
    layout = dict(title=f"{res['name']}: topicurile {list(dims)}", height=720,
                  legend=dict(itemsizing="constant"), margin=dict(l=0, r=0, t=50, b=0))
    if three:
        layout["scene"] = dict(xaxis=ax(dims[0]), yaxis=ax(dims[1]), zaxis=ax(dims[2]))
    else:
        layout.update(xaxis=ax(dims[0]), yaxis=ax(dims[1]))
    fig.update_layout(**layout)
    return fig


figs = {}
for name, res in results.items():
    for dims in DIM_SETS[res["k"]]:
        fig = plot_scatter(res, dims)
        key = f"{name} {'x'.join(map(str, dims))}"
        figs[key] = fig
        fn = re.sub(r"[^a-z0-9]+", "_", key.lower()).strip("_") + ".html"
        fig.write_html(OUT_DIR / "plots" / fn)
print(f"{len(figs)} grafice salvate în {OUT_DIR / 'plots'}")

10 grafice salvate în shared\plots


In [10]:
if SHOW_PLOTS:
    for key, fig in figs.items():
        fig.show()

## 6. Coerența Gensim

Cu doar 30 de documente, corpusul de referință este făcut din pseudo-paragrafe (bucăți de `CHUNK_SIZE` tokeni), aceleași pe care le primește și persoana B în `shared/`. Diferențele mici dintre modele sunt zgomot.

Notă: la LSA topicurile au semne mixte, iar aici se evaluează doar cuvintele cu încărcare *pozitivă*, deci coerența e mai puțin semnificativă decât la NMF/LDA.

In [11]:
def coherence_report(topics):
    topics = [[w for w in t if w in dictionary.token2id] for t in topics]
    scores = {}
    for m in ("c_v", "u_mass", "c_uci", "c_npmi"):
        cm = CoherenceModel(topics=topics, texts=chunk_texts, dictionary=dictionary,
                            coherence=m, processes=1)
        scores[m] = float(np.mean(cm.get_coherence_per_topic()))
    return scores


def topic_diversity(topics):
    words = [w for t in topics for w in t]
    return len(set(words)) / len(words)


rows = []
for name, res in results.items():
    tw = topic_words(res)
    rows.append({"model": name, **coherence_report(tw), "diversity": topic_diversity(tw)})
coh = pd.DataFrame(rows).set_index("model")
coh.round(3)

,c_v,u_mass,c_uci,c_npmi,diversity
model,,,,,
"LSA (BoW, k=3)",0.600,-4.848,-2.626,-0.030,0.90
"LSA (TF-IDF, k=3)",0.529,-7.873,-6.191,-0.168,0.90
"LSA (BoW, k=5)",0.541,-6.213,-3.732,-0.074,0.68
"LSA (TF-IDF, k=5)",0.640,-6.266,-4.271,-0.051,0.86


In [12]:
# Tabel final (coerență + atribuire) și export pentru Partea 4 / pentru unirea cu rezultatele lui B
final = coh.join(assign_summary)
final.to_csv(OUT_DIR / "results_A.csv")
(OUT_DIR / "lsa_topics_A.json").write_text(
    json.dumps({n: topic_words(r) for n, r in results.items()}, indent=2), encoding="utf-8")
final.round(3)

,c_v,u_mass,c_uci,c_npmi,diversity,purity (abs),ARI (abs),cel mai mare grup (abs),purity (kmeans),ARI (kmeans)
model,,,,,,,,,,
"LSA (BoW, k=3)",0.600,-4.848,-2.626,-0.030,0.90,0.933,0.808,12,1.0,1.000
"LSA (TF-IDF, k=3)",0.529,-7.873,-6.191,-0.168,0.90,1.000,1.000,10,1.0,1.000
"LSA (BoW, k=5)",0.541,-6.213,-3.732,-0.074,0.68,0.967,0.745,11,1.0,0.870
"LSA (TF-IDF, k=5)",0.640,-6.266,-4.271,-0.051,0.86,1.000,0.785,10,1.0,0.714


## Partea 4 (doar pentru LSA): răspunsuri de completat

Răspundeți folosind tabelele și graficele de mai sus; răspunsurile despre NMF/LDA le scrie persoana B, apoi le unificați în întâlnirea comună.

**Q1. BoW vs TF-IDF.** Cum diferă topicurile LSA (cuvinte, coerență, puritate) între BoW și TF-IDF? Care reprezentare dă topicuri mai curate și de ce (ponderea cuvintelor frecvente în BoW vs. penalizarea lor în TF-IDF)?

*(răspuns)*

**Q2. Interpretabilitate.** Sunt topicurile LSA ușor de interpretat? Ce efect au valorile negative și componenta 0 (dominată de lungimea documentului)? *(decizia "cea mai interpretabilă metodă" se ia împreună cu B)*

*(răspuns)*

**Q3. Cv mare vs. topicuri utile.** Găsiți un exemplu din tabelul de coerență în care un c_v mai mare nu înseamnă topicuri mai bune (de exemplu topicuri care repetă aceleași cuvinte sau care amestecă domenii).

*(răspuns)*

**Q4. Efectul lui k.** Comparați k=3 și k=5 pe LSA: ce se întâmplă cu coerența, cu diversitatea și cu puritatea? Ce reprezintă topicurile suplimentare (subdomenii? zgomot?).

*(răspuns)*

**Q5. Corespondența cu cele 3 domenii.** Câte dintre cele 3 domenii se regăsesc clar în topicurile LSA? Ce articole sunt atribuite "greșit" și de ce (ex.: articole de graniță)?

*(răspuns)*